# BAB 10 - Model Inference

Tahap selanjutnya adalah **Model Inference**. Tahap ini merupakan **proses penggunaan model yang telah dilatih** untuk melakukan prediksi pada data baru.

## Import Library

In [1]:
# Import libraries

import pickle
import pandas as pd
import numpy as np

## Load Model

In [2]:
''' 
  Load model terbaik, frequency encoding (country_name_freq),
  dan preprocessing objects (median_imputer) yang filenya telah 
  dibuat di tahap sebelumnya.
'''

with open('best_model.pkl', 'rb') as file_best_model:
  best_model = pickle.load(file_best_model)

# frequency encoding 
with open('country_name_freq.pkl', 'rb') as file_country_name_freq:
  freq_country_name = pickle.load(file_country_name_freq)

# preprocessing objects
with open('median_imputer.pkl', 'rb') as file_median_imputer:
  median_imputer = pickle.load(file_median_imputer)

In [3]:
''' 
    Membuat fungsi preprocess_input yang digunakan untuk 
    melakukan preprocessing data baru sebelum melakukan prediksi. 
    Tahapan yang dilakukan meliputi penanganan missing value 
    menggunakan median imputer, penanganan outlier dengan metode 
    z-score & IQR serta frequency encoding kategorikal.  
'''

def preprocess_input(df):
    df = df.copy()

    # Frequency Encoding
    df['country_name_freq'] = df['country_name'].map(freq_country_name).fillna(0)
    df.drop(columns=['country_name'], inplace=True)

    # Missing Value (median dari training)
    num_cols_impute = [
        'adolescent_fertility_rate_births_per_1000_women_ages_15-19',
        'age_dependency_ratio_pct_of_working-age_population',
        'death_rate_crude_per_1000_people',
        'fertility_rate_total_births_per_woman',
        'mortality_rate_infant_per_1000_live_births',
        'mortality_rate_under-5_per_1000',
        'population_ages_0-14_pct_of_total_population',
        'population_ages_65_and_above_pct_of_total_population',
        'population_growth_annual_pct',
        'urban_population_pct_of_total_population'
    ]

    df[num_cols_impute] = median_imputer.transform(df[num_cols_impute])

    # Outlier Handling 
    # 1. Normal
    for col in [    
        'year', 'age_dependency_ratio_pct_of_working-age_population',
        'fertility_rate_total_births_per_woman', 'population_ages_0-14_pct_of_total_population',
        'urban_population_pct_of_total_population'
    ]:
        avg = df[col].mean()
        std = df[col].std()
        
        lower =  avg - 3 * std
        upper = avg + 3 * std
        
        df[col] = df[col].clip(lower, upper)

    # 2. Skew
    for col in [    
        'adolescent_fertility_rate_births_per_1000_women_ages_15-19'
    ]:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        df[col] = df[col].clip(lower, upper)

    # 3. Extreme Skew
    for col in [    'death_rate_crude_per_1000_people', 'mortality_rate_infant_per_1000_live_births', 
                    'mortality_rate_under-5_per_1000', 'population_ages_65_and_above_pct_of_total_population',
                    'population_growth_annual_pct'
    ]:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 3 * IQR
        upper = Q3 + 3 * IQR
        
        df[col] = df[col].clip(lower, upper)

    return df

## Inferencing

In [4]:
''' 
    Data baru yang diinput oleh negara Indonesia tahun 2023.
'''
ind_2023 = {
    'country_name': 'Indonesia',
    'year': 2023, 
    'region': 'South Asia', 
    'income_group': 'Lower middle income',
    'adolescent_fertility_rate_births_per_1000_women_ages_15-19':150,
    'age_dependency_ratio_pct_of_working-age_population':89,
    'current_health_expenditure_per_capita_current_us$':100,
    'death_rate_crude_per_1000_people':40,
    'fertility_rate_total_births_per_woman':7.5,
    'hospital_beds_per_1000_people':0.2,
    'mortality_rate_infant_per_1000_live_births':120,
    'mortality_rate_under-5_per_1000':120,
    'nurses_and_midwives_per_1000_people':0.14,
    'people_using_safely_managed_drinking_water_services_pct_of_population':25,
    'people_using_safely_managed_sanitation_services_pct_of_population':37.5,
    'people_with_basic_handwashing_facilities_including_soap_and_water_pct_of_population':42.5,
    'physicians_per_1000_people':0.25,
    'population_ages_0-14_pct_of_total_population':44.77,
    'population_ages_65_and_above_pct_of_total_population':4.56,
    'population_growth_annual_pct':1.67,
    'prevalence_of_stunting_height_for_age_pct_of_children_under_5':25,
    'prevalence_of_undernourishment_pct_of_population':40,
    'urban_population_pct_of_total_population':36.99
}

ind_2023 = pd.DataFrame([ind_2023])
ind_2023

,country_name,year,region,income_group,adolescent_fertility_rate_births_per_1000_women_ages_15-19,age_dependency_ratio_pct_of_working-age_population,current_health_expenditure_per_capita_current_us$,death_rate_crude_per_1000_people,fertility_rate_total_births_per_woman,hospital_beds_per_1000_people,...,people_using_safely_managed_drinking_water_services_pct_of_population,people_using_safely_managed_sanitation_services_pct_of_population,people_with_basic_handwashing_facilities_including_soap_and_water_pct_of_population,physicians_per_1000_people,population_ages_0-14_pct_of_total_population,population_ages_65_and_above_pct_of_total_population,population_growth_annual_pct,prevalence_of_stunting_height_for_age_pct_of_children_under_5,prevalence_of_undernourishment_pct_of_population,urban_population_pct_of_total_population
0,Indonesia,2023,South Asia,Lower middle income,150,89,100,40,7.5,0.2,...,25,37.5,42.5,0.25,44.77,4.56,1.67,25,40,36.99


In [5]:
''' 
    Data baru yang diinput oleh negara Indonesia tahun 2025.
'''
ind_2025 = {
    'country_name': 'Indonesia',
    'year': 2025, 
    'region': 'South Asia', 
    'income_group': 'Lower middle income',
    'adolescent_fertility_rate_births_per_1000_women_ages_15-19':150,
    'age_dependency_ratio_pct_of_working-age_population':89,
    'current_health_expenditure_per_capita_current_us$':200,
    'death_rate_crude_per_1000_people':0.4,
    'fertility_rate_total_births_per_woman':5,
    'hospital_beds_per_1000_people':0.5,
    'mortality_rate_infant_per_1000_live_births':12,
    'mortality_rate_under-5_per_1000':12,
    'nurses_and_midwives_per_1000_people':0.5,
    'people_using_safely_managed_drinking_water_services_pct_of_population':50,
    'people_using_safely_managed_sanitation_services_pct_of_population':56.88,
    'people_with_basic_handwashing_facilities_including_soap_and_water_pct_of_population':64.22,
    'physicians_per_1000_people':0.66,
    'population_ages_0-14_pct_of_total_population':44.77,
    'population_ages_65_and_above_pct_of_total_population':8.56,
    'population_growth_annual_pct':2.67,
    'prevalence_of_stunting_height_for_age_pct_of_children_under_5':5,
    'prevalence_of_undernourishment_pct_of_population':10,
    'urban_population_pct_of_total_population':36.99
}

ind_2025 = pd.DataFrame([ind_2025])
ind_2025

,country_name,year,region,income_group,adolescent_fertility_rate_births_per_1000_women_ages_15-19,age_dependency_ratio_pct_of_working-age_population,current_health_expenditure_per_capita_current_us$,death_rate_crude_per_1000_people,fertility_rate_total_births_per_woman,hospital_beds_per_1000_people,...,people_using_safely_managed_drinking_water_services_pct_of_population,people_using_safely_managed_sanitation_services_pct_of_population,people_with_basic_handwashing_facilities_including_soap_and_water_pct_of_population,physicians_per_1000_people,population_ages_0-14_pct_of_total_population,population_ages_65_and_above_pct_of_total_population,population_growth_annual_pct,prevalence_of_stunting_height_for_age_pct_of_children_under_5,prevalence_of_undernourishment_pct_of_population,urban_population_pct_of_total_population
0,Indonesia,2025,South Asia,Lower middle income,150,89,200,0.4,5,0.5,...,50,56.88,64.22,0.66,44.77,8.56,2.67,5,10,36.99


## Predict the Data

In [6]:
'''
    Memanggil function preprocess input pada ind_2023 
    lalu memprediksi hasil dan menghitung probabilitas.
'''
ind_2023_final = preprocess_input(ind_2023)

y_pred = best_model.predict(ind_2023_final)
y_proba = best_model.predict_proba(ind_2023_final)

prob_selected = y_proba[0][y_pred[0]]

print(f'Prediction: {y_pred[0]}')
print(f'Probability: {prob_selected:.0%}')

if y_pred[0] == 1:
    print("\nHealth Status: High")
else:
    print("\nHealth Status: Low")

Prediction: 0
Probability: 98%

Health Status: Low


Berdasarkan hasil *inference*, model memprediksi bahwa **negara Indonesia tahun 2023 termasuk dalam kategori 0 atau Low** dengan tingkat kepercayaan sebesar **98%**. Hal ini menunjukkan bahwa model memiliki keyakinan yang tinggi terhadap hasil prediksi tersebut.

In [7]:
'''
    Memanggil function preprocess input pada ind_2025 
    lalu memprediksi hasil dan menghitung probabilitas.
'''
ind_2025_final = preprocess_input(ind_2025)

y_pred = best_model.predict(ind_2025_final)
y_proba = best_model.predict_proba(ind_2025_final)

prob_selected = y_proba[0][y_pred[0]]

print(f'Prediction: {y_pred[0]}')
print(f'Probability: {prob_selected:.0%}')

if y_pred[0] == 1:
    print("\nHealth Status: High")
else:
    print("\nHealth Status: Low")

Prediction: 1
Probability: 75%

Health Status: High


Berdasarkan hasil *inference*, model memprediksi bahwa **negara Indonesia tahun 2025 termasuk dalam kategori 1 atau High** dengan tingkat kepercayaan sebesar **75%**. Hal ini menunjukkan bahwa model memiliki keyakinan yang cukup terhadap hasil prediksi tersebut.